# Introduction to MediaDir

This notebook demonstrates the MediaDir functionality from `mediatools`, which provides a powerful recursive data structure for organizing and managing media files in filesystem directories.

**What you'll learn:**
- How to scan directories for media files (videos, images, other files)
- Navigate and work with nested directory structures
- Access and process media files at individual and batch levels
- Compare directories and detect changes
- Integration patterns with other mediatools modules

**Prerequisites:**
- Basic Python knowledge
- Familiarity with pathlib for file system operations
- Some media files to experiment with (we'll create mock examples)

**Key Concepts:**
- **MediaDir**: Recursive directory structure containing videos, images, and subdirectories
- **File Collections**: VideoFiles, ImageFiles, and other file containers
- **Metadata Integration**: Each file type provides rich metadata access
- **Path Flexibility**: Support for both absolute and relative path handling

## Import Section

In [ ]:
import mediatools
import pathlib
import tempfile
import shutil
from pathlib import Path

## Setup and Mock Data Creation

Since this is a demonstration notebook, we'll create a realistic directory structure with mock media files. This allows you to run the examples without needing your own media collection.

In [ ]:
# Create a temporary directory structure with mock media files
def create_mock_media_directory():
    """Create a realistic directory structure with mock media files for demonstration"""
    
    # Create temporary directory
    temp_dir = Path(tempfile.mkdtemp(prefix="mediatools_demo_"))
    print(f"Created demo directory: {temp_dir}")
    
    # Create directory structure
    directories = [
        temp_dir / "vacation_2023",
        temp_dir / "vacation_2023" / "beach", 
        temp_dir / "vacation_2023" / "mountains",
        temp_dir / "family_events",
        temp_dir / "family_events" / "birthday",
        temp_dir / "projects" / "timelapse",
    ]
    
    for directory in directories:
        directory.mkdir(parents=True, exist_ok=True)
    
    # Create mock video files (empty files with correct extensions)
    video_files = [
        temp_dir / "vacation_2023" / "beach" / "surfing.mp4",
        temp_dir / "vacation_2023" / "beach" / "sunset.mov", 
        temp_dir / "vacation_2023" / "mountains" / "hiking.mp4",
        temp_dir / "vacation_2023" / "mountains" / "drone_footage.mp4",
        temp_dir / "family_events" / "birthday" / "cake_ceremony.mp4",
        temp_dir / "family_events" / "birthday" / "kids_playing.avi",
        temp_dir / "projects" / "timelapse" / "construction.mkv",
    ]
    
    # Create mock image files
    image_files = [
        temp_dir / "vacation_2023" / "beach" / "group_photo.jpg",
        temp_dir / "vacation_2023" / "beach" / "shells.png",
        temp_dir / "vacation_2023" / "mountains" / "panorama.jpg",
        temp_dir / "vacation_2023" / "mountains" / "wildlife.jpg",
        temp_dir / "family_events" / "birthday" / "decorations.jpg",
        temp_dir / "family_events" / "family_portrait.png",
        temp_dir / "projects" / "timelapse" / "setup.jpg",
    ]
    
    # Create mock other files
    other_files = [
        temp_dir / "vacation_2023" / "itinerary.txt",
        temp_dir / "family_events" / "guest_list.xlsx",
        temp_dir / "projects" / "README.md",
    ]
    
    # Write empty files
    all_files = video_files + image_files + other_files
    for file_path in all_files:
        file_path.touch()
    
    return temp_dir

# Create our demo directory
demo_root = create_mock_media_directory()

print("\nDemo directory structure created:")
for item in sorted(demo_root.rglob("*")):
    if item.is_file():
        print(f"  {item.relative_to(demo_root)}")

## Basic MediaDir Usage

### Creating a MediaDir from a Directory

The primary way to create a MediaDir is by scanning a filesystem directory. MediaDir will automatically categorize files into videos, images, and other files based on their extensions.

In [ ]:
# Method 1: Using the scan_directory function (recommended)
media_dir = mediatools.scan_directory(demo_root)

print(f"MediaDir created for: {media_dir.path}")
print(f"Contains {len(media_dir.subdirs)} subdirectories")
print(f"Contains {len(media_dir.videos)} videos in root")
print(f"Contains {len(media_dir.images)} images in root")
print(f"Contains {len(media_dir.other_files)} other files in root")

In [ ]:
# Method 2: Using the MediaDir class method directly
media_dir_alt = mediatools.MediaDir.from_path(demo_root)

# Both methods produce equivalent results
print(f"Same result: {media_dir.path == media_dir_alt.path}")

### Exploring Directory Contents

MediaDir provides intuitive properties to access different types of content within a directory.

In [ ]:
# Access subdirectories
print("Subdirectories:")
for subdir_name, subdir in media_dir.subdirs.items():
    print(f"  {subdir_name}: {subdir.path}")
    
print("\nFiles in root directory:")
print(f"Videos: {[v.path.name for v in media_dir.videos]}")
print(f"Images: {[i.path.name for i in media_dir.images]}")
print(f"Other files: {[f.path.name for f in media_dir.other_files]}")

## Working with Nested Directories

### Navigation Methods

MediaDir provides several convenient ways to navigate through nested directory structures.

In [ ]:
# Method 1: Using subdir() method
vacation_dir = media_dir.subdir("vacation_2023")
beach_dir = media_dir.subdir("vacation_2023", "beach")

print(f"Vacation directory: {vacation_dir.path}")
print(f"Beach directory: {beach_dir.path}")

# Method 2: Using bracket notation
mountains_dir = media_dir["vacation_2023"]["mountains"]
print(f"Mountains directory: {mountains_dir.path}")

# Method 3: Using pathlib-style path
birthday_dir = media_dir.subdir(Path("family_events") / "birthday")
print(f"Birthday directory: {birthday_dir.path}")

In [ ]:
# Explore content in nested directories
print("Beach directory contents:")
print(f"  Videos: {[v.path.name for v in beach_dir.videos]}")
print(f"  Images: {[i.path.name for i in beach_dir.images]}")

print("\nMountains directory contents:")
print(f"  Videos: {[v.path.name for v in mountains_dir.videos]}")
print(f"  Images: {[i.path.name for i in mountains_dir.images]}")

### Parent Directory Navigation

In [ ]:
# Get parent hierarchy
parents = beach_dir.parents()
print("Parent hierarchy (from deepest to root):")
for i, parent in enumerate(parents):
    indent = "  " * i
    print(f"{indent}{parent.path.name} -> {parent.path}")

# Direct parent access
print(f"\nBeach directory parent: {beach_dir.parent.path}")
print(f"Root directory: {beach_dir.root.path}")

## Recursive File Access

### Getting All Files Across the Entire Directory Tree

One of MediaDir's most powerful features is the ability to collect all files recursively across the entire directory structure.

In [ ]:
# Get all video files recursively
all_videos = media_dir.all_video_files()
print(f"Total videos found: {len(all_videos)}")
print("All video files:")
for video in all_videos:
    rel_path = video.path.relative_to(demo_root)
    print(f"  {rel_path}")

print(f"\nTotal images found: {len(media_dir.all_image_files())}")
print("All image files:")
for image in media_dir.all_image_files():
    rel_path = image.path.relative_to(demo_root)
    print(f"  {rel_path}")

In [ ]:
# Get all file paths (including non-media files)
all_file_paths = media_dir.all_file_paths()
media_file_paths = media_dir.all_media_paths()

print(f"All files: {len(all_file_paths)}")
print(f"Media files only: {len(media_file_paths)}")
print(f"Non-media files: {len(all_file_paths) - len(media_file_paths)}")

## Advanced File Retrieval

### Finding Specific Files by Path

In [ ]:
# Find a specific video file by path
video_path = demo_root / "vacation_2023" / "beach" / "surfing.mp4"
video_file = media_dir.get_video(video_path)
print(f"Found video: {video_file.path}")

# Find a specific image file
image_path = demo_root / "vacation_2023" / "mountains" / "panorama.jpg"
image_file = media_dir.get_image(image_path)
print(f"Found image: {image_file.path}")

# Find a non-media file
other_path = demo_root / "vacation_2023" / "itinerary.txt"
other_file = media_dir.get_nonmedia(other_path)
print(f"Found other file: {other_file.path}")

## Configuration Options

### Custom File Extensions and Filtering

In [ ]:
# Create MediaDir with custom file extensions
custom_media_dir = mediatools.scan_directory(
    demo_root,
    video_ext=['.mp4', '.mov'],  # Only MP4 and MOV files
    image_ext=['.jpg', '.jpeg']  # Only JPEG files
)

print("With custom extensions:")
print(f"Videos found: {len(custom_media_dir.all_video_files())}")
print(f"Images found: {len(custom_media_dir.all_image_files())}")

# Compare with default (more permissive) scan
print("\nWith default extensions:")
print(f"Videos found: {len(media_dir.all_video_files())}")
print(f"Images found: {len(media_dir.all_image_files())}")

In [ ]:
# Create MediaDir with path filtering
def ignore_timelapse_projects(path):
    """Ignore any paths containing 'timelapse' or 'projects'"""
    return 'timelapse' in str(path) or 'projects' in str(path)

filtered_media_dir = mediatools.scan_directory(
    demo_root,
    ignore_path=ignore_timelapse_projects
)

print("With path filtering (ignoring timelapse/projects):")
print(f"Videos found: {len(filtered_media_dir.all_video_files())}")
print(f"Subdirectories: {list(filtered_media_dir.subdirs.keys())}")

In [ ]:
# Using relative paths instead of absolute
relative_media_dir = mediatools.scan_directory(
    demo_root,
    use_absolute=False
)

print("Path handling comparison:")
print(f"Absolute path: {media_dir.path}")
print(f"Relative path: {relative_media_dir.path}")

## Directory Comparison and Synchronization

### Detecting Changes Between Directory States

In [ ]:
# Create a modified version of our directory for comparison
def create_modified_directory():
    """Create a slightly modified version of our demo directory"""
    modified_root = Path(tempfile.mkdtemp(prefix="mediatools_modified_"))
    
    # Copy the original structure
    shutil.copytree(demo_root, modified_root, dirs_exist_ok=True)
    
    # Make some changes:
    # 1. Remove a file
    (modified_root / "vacation_2023" / "beach" / "shells.png").unlink()
    
    # 2. Add a new file
    (modified_root / "vacation_2023" / "new_video.mp4").touch()
    
    # 3. Add a new directory with files
    new_dir = modified_root / "vacation_2023" / "city"
    new_dir.mkdir()
    (new_dir / "architecture.jpg").touch()
    (new_dir / "street_performance.mp4").touch()
    
    return modified_root

# Create modified directory and MediaDir
modified_root = create_modified_directory()
modified_media_dir = mediatools.scan_directory(modified_root)

print(f"Original directory files: {len(media_dir.all_file_paths())}")
print(f"Modified directory files: {len(modified_media_dir.all_file_paths())}")

In [ ]:
# Compare the two directory structures
removed_files, added_files = media_dir.file_diff(modified_media_dir)

print("Files removed in modified version:")
for removed in removed_files:
    print(f"  - {removed}")

print("\nFiles added in modified version:")
for added in added_files:
    print(f"  + {added}")

In [ ]:
# Find directories that have changes
changed_dirs = media_dir.get_changed_dirs(modified_media_dir)

print("Directories with changes:")
for changed_dir in changed_dirs:
    rel_path = changed_dir.path.relative_to(demo_root)
    print(f"  {rel_path}")
    
    # Show what changed in each directory
    dir_removed, dir_added = changed_dir.file_diff(
        modified_media_dir.subdir(rel_path) if rel_path != Path('.') 
        else modified_media_dir
    )
    
    if dir_removed:
        print(f"    Removed: {[f.name for f in dir_removed]}")
    if dir_added:
        print(f"    Added: {[f.name for f in dir_added]}")

## Integration with Other MediaTools Modules

### Working with VideoFile and ImageFile Objects

In [ ]:
# Access individual file objects and their capabilities
print("Video file integration examples:")
for video_file in media_dir.all_video_files()[:2]:  # First 2 videos
    print(f"\n{video_file.path.name}:")
    print(f"  Type: {type(video_file).__name__}")
    print(f"  Path: {video_file.path}")
    
    # Note: These methods would work with real video files
    # For demo purposes, we just show the API
    print(f"  Available methods: probe(), read_meta(), ffmpeg(), etc.")

print("\n" + "="*50)
print("Image file integration examples:")
for image_file in media_dir.all_image_files()[:2]:  # First 2 images
    print(f"\n{image_file.path.name}:")
    print(f"  Type: {type(image_file).__name__}")
    print(f"  Path: {image_file.path}")
    
    # Note: These methods would work with real image files
    print(f"  Available methods: read(), read_meta(), etc.")

## Serialization and Data Export

### Converting MediaDir to Dictionary Formats

In [ ]:
# Convert to dictionary representation
media_dict = media_dir.to_dict()

print("MediaDir dictionary representation (top level):")
for key, value in media_dict.items():
    if key == 'subdirs':
        print(f"  {key}: {list(value.keys()) if isinstance(value, dict) else value}")
    elif isinstance(value, list):
        print(f"  {key}: [{len(value)} items]")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Convert to file tree representation (simpler structure)
file_tree = media_dir.to_file_tree()

def print_file_tree(tree, indent=0):
    """Recursively print the file tree structure"""
    for key, value in tree.items():
        print("  " * indent + key)
        if isinstance(value, dict):
            print_file_tree(value, indent + 1)

print("File tree representation:")
print_file_tree(file_tree)

## Practical Use Cases and Workflows

### 1. Media Library Organization

In [ ]:
def analyze_media_library(media_dir):
    """Generate a comprehensive analysis of a media library"""
    
    analysis = {
        'total_directories': len(list(media_dir.path.rglob('*'))) if media_dir.path.exists() else 0,
        'total_videos': len(media_dir.all_video_files()),
        'total_images': len(media_dir.all_image_files()),
        'total_other_files': len(media_dir.all_file_paths()) - len(media_dir.all_media_paths()),
        'directory_breakdown': {}
    }
    
    # Analyze each subdirectory
    for subdir_name, subdir in media_dir.subdirs.items():
        analysis['directory_breakdown'][subdir_name] = {
            'videos': len(subdir.all_video_files()),
            'images': len(subdir.all_image_files()),
            'subdirectories': len(subdir.subdirs)
        }
    
    return analysis

# Analyze our demo library
analysis = analyze_media_library(media_dir)

print("Media Library Analysis:")
print(f"Total Videos: {analysis['total_videos']}")
print(f"Total Images: {analysis['total_images']}")
print(f"Total Other Files: {analysis['total_other_files']}")
print("\nDirectory Breakdown:")
for dir_name, stats in analysis['directory_breakdown'].items():
    print(f"  {dir_name}: {stats['videos']} videos, {stats['images']} images, {stats['subdirectories']} subdirs")

### 2. Batch Processing Preparation

In [ ]:
def prepare_batch_processing_list(media_dir, file_type='video'):
    """Prepare a list of files for batch processing with organized metadata"""
    
    if file_type == 'video':
        files = media_dir.all_video_files()
    elif file_type == 'image':
        files = media_dir.all_image_files()
    else:
        raise ValueError("file_type must be 'video' or 'image'")
    
    processing_list = []
    for file_obj in files:
        # Get relative path for organization
        rel_path = file_obj.path.relative_to(media_dir.path)
        
        # Determine category based on directory structure
        path_parts = rel_path.parts
        category = path_parts[0] if len(path_parts) > 1 else 'root'
        subcategory = path_parts[1] if len(path_parts) > 2 else None
        
        processing_list.append({
            'file_object': file_obj,
            'full_path': file_obj.path,
            'relative_path': rel_path,
            'category': category,
            'subcategory': subcategory,
            'filename': file_obj.path.name,
            'extension': file_obj.path.suffix.lower()
        })
    
    return processing_list

# Prepare video processing list
video_batch = prepare_batch_processing_list(media_dir, 'video')

print("Video files prepared for batch processing:")
for item in video_batch:
    print(f"  {item['category']}/{item['subcategory'] or ''}: {item['filename']} ({item['extension']})")

# Group by category for organized processing
by_category = {}
for item in video_batch:
    category = item['category']
    if category not in by_category:
        by_category[category] = []
    by_category[category].append(item)

print("\nGrouped by category:")
for category, items in by_category.items():
    print(f"  {category}: {len(items)} files")

### 3. Change Detection and Backup Verification

In [ ]:
def create_backup_verification_report(source_dir, backup_dir):
    """Create a detailed report comparing source and backup directories"""
    
    removed, added = source_dir.file_diff(backup_dir)
    
    report = {
        'backup_complete': len(removed) == 0 and len(added) == 0,
        'missing_from_backup': removed,
        'extra_in_backup': added,
        'source_stats': {
            'total_files': len(source_dir.all_file_paths()),
            'videos': len(source_dir.all_video_files()),
            'images': len(source_dir.all_image_files())
        },
        'backup_stats': {
            'total_files': len(backup_dir.all_file_paths()),
            'videos': len(backup_dir.all_video_files()),
            'images': len(backup_dir.all_image_files())
        }
    }
    
    return report

# Compare original with our modified directory (simulating backup check)
verification = create_backup_verification_report(media_dir, modified_media_dir)

print("Backup Verification Report:")
print(f"Backup Complete: {verification['backup_complete']}")
print(f"\nSource Directory: {verification['source_stats']['total_files']} total files")
print(f"Backup Directory: {verification['backup_stats']['total_files']} total files")

if verification['missing_from_backup']:
    print(f"\nMissing from backup ({len(verification['missing_from_backup'])} files):")
    for missing in list(verification['missing_from_backup'])[:5]:  # Show first 5
        print(f"  - {missing}")

if verification['extra_in_backup']:
    print(f"\nExtra files in backup ({len(verification['extra_in_backup'])} files):")
    for extra in list(verification['extra_in_backup'])[:5]:  # Show first 5
        print(f"  + {extra}")

## Cleanup

Let's clean up our temporary directories.

In [ ]:
# Clean up temporary directories
import shutil

try:
    shutil.rmtree(demo_root)
    shutil.rmtree(modified_root)
    print("Temporary directories cleaned up successfully.")
except Exception as e:
    print(f"Cleanup warning: {e}")

## Summary

This notebook demonstrated the core functionality of the MediaDir system:

**Key Features Covered:**
- **Directory Scanning**: Automatically categorize media files by type
- **Recursive Navigation**: Easy access to nested directory structures
- **File Collections**: Organized access to videos, images, and other files
- **Batch Operations**: Process all files across directory trees
- **Change Detection**: Compare directory states for synchronization
- **Flexible Configuration**: Custom extensions and filtering options

**Integration Points:**
- VideoFile objects provide FFmpeg integration for video processing
- ImageFile objects enable image manipulation and analysis
- Serialization support for persistence and data export
- Path flexibility for both development and production scenarios

**Common Use Cases:**
1. **Media Library Management**: Organize and catalog large collections
2. **Batch Processing**: Process files systematically across directory trees
3. **Backup Verification**: Ensure completeness of media backups
4. **Change Monitoring**: Detect additions, removals, and modifications
5. **Website Generation**: Build galleries from filesystem structure

MediaDir serves as the foundational data structure that makes complex media workflows simple and intuitive, providing the organizational backbone for the entire mediatools ecosystem.